# Mise à jour des adresses de test


## Imports

In [1]:
import os
from datetime import datetime
from pathlib import Path

import polars as pl
from IPython.display import display
from polars.dataframe.frame import DataFrame

## Variables

In [2]:
EXCEL_PATH = r"C:\Users\denis.iglesias\OneDrive - HESSO\01 Institution\02 Projets\00 Archive\03 ECS-Optiwatt eco21-SIG"

In [3]:
# search in EXCEL_PATH in all its sub-folders for the files that are excel
excel_files: list[str] = []
for root, dirs, files in os.walk(EXCEL_PATH):
    for filename in files:
        if filename.endswith(".xlsx"):
            excel_files.append(os.path.join(root, filename))

# Remove certain paths that contains certain keywords
exclude_keywords: list[str] = ["99 Ventilation", "06_cours_python", "sitg"]
excel_files: list[str] = [
    f for f in excel_files if not any(keyword in f for keyword in exclude_keywords)
]

display(f"Found {len(excel_files)} files in {EXCEL_PATH}")
display(excel_files)

'Found 107 files in C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG'

['C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\00 Ancien\\Base_donnees v5.xlsx',
 'C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\00 Ancien\\Base_donnees v6.xlsx',
 'C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\00 Ancien\\ECO21_ECS_PME_Analyse disperison et moyennes v2_EBP.xlsx',
 'C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\00 Ancien\\ECO21_ECS_PME_Analyse disperison et moyennes v3_EBP.xlsx',
 'C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\00 Ancien\\Tableau de suivi ECS Optiwatt 2019-20 v11.xlsx',
 'C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\00 Ancien\\Tableau de suivi ECS Optiwatt 2019-20

In [4]:
"""
Read every sheet of every Excel file, auto-detecting the header row per sheet
(since it is not always the first row), then concatenate everything with
diagonal_relaxed (union of columns, nulls for missing, common supertype for
mismatched dtypes).
"""


def is_sequential_index_row(row: tuple) -> bool:
    """
    Detect a manually-typed row of sequential column numbers (1, 2, 3, ...)
    sometimes placed above the real header as a numbering aid. Such a row is
    fully populated and all-unique, so it otherwise passes as a false-positive
    header.
    """
    try:
        as_ints = [int(v) for v in row]
    except (TypeError, ValueError):
        return False
    return as_ints == list(range(1, len(row) + 1))


def detect_header_row(raw: pl.DataFrame, max_scan_rows: int = 15) -> int:
    """
    Guess which row (0-indexed) holds the column headers.
    Heuristic: a header row is fully populated (no nulls) and has no
    duplicate values, since real headers are distinct labels. We scan the
    first `max_scan_rows` rows and pick the first one matching that pattern,
    skipping sequential numbering rows (1, 2, 3, ...) that would otherwise
    look like a valid header.
    Falls back to row 0 if nothing matches (e.g. sheet truly has no junk rows).
    """
    n_scan = min(max_scan_rows, raw.height)
    for i in range(n_scan):
        row = raw.row(i)
        non_null = [v for v in row if v is not None]
        if (
            len(non_null) == len(row)
            and len(set(non_null)) == len(non_null)
            and not is_sequential_index_row(row)
        ):
            return i
    return 0


def promote_header(raw: pl.DataFrame, header_idx: int) -> pl.DataFrame:
    """Rename columns using the detected header row, drop rows up to and including it."""
    header_values = raw.row(header_idx)
    # Fallback names for any blank header cells, avoid duplicate names
    seen: dict[str, int] = {}
    new_names = []
    for i, v in enumerate(header_values):
        name = str(v) if v is not None else f"col_{i}"
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        new_names.append(name)

    df = raw.slice(header_idx + 1, raw.height - header_idx - 1)
    df.columns = new_names
    return df


def read_raw_sheets(path: str) -> dict[str, pl.DataFrame]:
    """
    Read all sheets of a file, raw (no header assumed).
    calamine is tried first (fast), falling back to openpyxl (slower, more
    tolerant) if calamine chokes on formula-error cells like #NAME?, #REF!, etc.
    raise_if_empty=False so genuinely empty sheets return an empty frame
    instead of raising NoDataError.
    """
    try:
        return pl.read_excel(
            source=path,
            sheet_id=0,
            engine="calamine",
            has_header=False,
            raise_if_empty=False,
        )
    except Exception as calamine_error:
        try:
            return pl.read_excel(
                source=path,
                sheet_id=0,
                engine="openpyxl",
                has_header=False,
                raise_if_empty=False,
            )
        except Exception as openpyxl_error:
            raise RuntimeError(
                f"calamine failed ({calamine_error}); "
                f"openpyxl fallback also failed ({openpyxl_error})"
            ) from openpyxl_error


def read_all_excel(paths: list[str]) -> list[pl.DataFrame]:
    """Read all sheets of all given Excel files, auto-fixing header row per sheet."""
    frames: list[pl.DataFrame] = []
    errors: list[tuple[str, str]] = []

    for path in paths:
        try:
            raw_sheets = read_raw_sheets(path)
        except Exception as e:
            # Include exception type so future failures are easier to diagnose at a glance
            errors.append((path, f"{type(e).__name__}: {e}"))
            continue

        for sheet_name, raw in raw_sheets.items():
            if raw.height == 0 or raw.width == 0:
                continue
            header_idx = detect_header_row(raw)
            df = promote_header(raw, header_idx)

            # Tag origin, including detected header row for traceability/debugging
            df = df.with_columns(
                pl.lit(Path(path).name).alias("source_file"),
                pl.lit(sheet_name).alias("source_sheet"),
                pl.lit(header_idx).alias("header_row_detected"),
            )
            frames.append(df)

    if errors:
        display(f"Failed to read {len(errors)} file(s):")
        for path, msg in errors:
            display(f"  {path}: {msg}")

    return frames


def concat_safely(frames: list[pl.DataFrame], batch_size: int = 20) -> pl.DataFrame:
    """
    Concatenate frames with diagonal_relaxed in batches.
    Batching avoids a known crash when column counts differ too much
    across a single very large concat call.
    """
    if not frames:
        raise ValueError("No frames to concatenate.")

    batches: list[pl.DataFrame] = []
    for i in range(0, len(frames), batch_size):
        chunk = frames[i : i + batch_size]
        batches.append(pl.concat(chunk, how="diagonal_relaxed"))

    return pl.concat(batches, how="diagonal_relaxed")


# --- Run ---
frames = read_all_excel(excel_files)
display(f"Loaded {len(frames)} sheet(s) from {len(excel_files)} file(s)")

# Sanity check: see which header rows were detected, worth a quick eyeball
header_summary = (
    pl.concat(
        [
            f.select("source_file", "source_sheet", "header_row_detected").head(1)
            for f in frames
        ],
        how="diagonal_relaxed",
    )
    if frames
    else None
)
display(header_summary)

df_all = concat_safely(frames)
display(f"Final shape: {df_all.shape}")
display(df_all)

c:\Users\denis.iglesias\Documents\GitHub\geocoder-service\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Could not determine dtype for column 90, falling back to string
Could not determine dtype for column 1, falling back to string
Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 1, falling back to string
Could not determine dtype for column 2, falling back to string
Could not determine dtype for column 90, falling back to string
Could not determine dtype for column 1, falling back to string
Could not determine dtype for column 3, falling back to string
Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 

'Failed to read 1 file(s):'

'  C:\\Users\\denis.iglesias\\OneDrive - HESSO\\01 Institution\\02 Projets\\00 Archive\\03 ECS-Optiwatt eco21-SIG\\02 Tableaux\\00 Ancien\\Tableau de suivi ECS Optiwatt 2019-20 v13.xlsx: RuntimeError: calamine failed (calamine cell error: #NAME?\nContext:\n    0: could not determine dtype for column __UNNAMED__44\n); openpyxl fallback also failed (Series name must be a string)'

'Loaded 509 sheet(s) from 107 file(s)'

source_file,source_sheet,header_row_detected
str,str,i32
"""Base_donnees v5.xlsx""","""1""",1
"""Base_donnees v5.xlsx""","""SITG_Adresse_EGID""",0
"""Base_donnees v5.xlsx""","""EGID_combustibles""",0
"""Base_donnees v5.xlsx""","""Chaudiere_annee""",0
"""Base_donnees v5.xlsx""","""EGID_chaudière""",0
…,…,…
"""df_02_merge_adresses.xlsx""","""Sheet1""",0
"""df_04_merge_analyse.xlsx""","""Sheet1""",0
"""DERNIER_INDICE.xlsx""","""Sheet1""",0


'Final shape: (5650758, 992)'

Numéro intervention,Installation,numéro partenaire,Raison sociale,Rue et numéro renseignés,rue SITG,Minergie,remarque,EGID bâtiment,Chaufferie dessert autres bâtiments,EGID_chaufferie,Année chaudière,Epoque,Agent_energetique_principal,IDC 2011,IDC 2012,IDC 2013,IDC 2014,IDC 2015,IDC 2016,IDC 2017,IDC 2018,IDC 2019,EF 2015 [m³],EF 2016 [m³],EF 2017 [m³],EF 2018 [m³],EF 2019 [m³],Année intervention,Activité,Dernier IDC,SRE [m²],Agent_énergétique1,Conso_agent_energetique1,Unite_agent_energetique1,conso_agent1 [kWh],Agent_énergétique2,…,rue_et_nb,cp,localite,secteur_dactivite,date_brt,douchette_prosecco_nb101602,douchette_variete_nb101601,regulateur_de_douche_nb500711,douchette_coiffeurs_express_lavabo_m22_ou_m24_nb100701,douche_murale,douche_gjosa,reducteur_de_debit_robinet_dia_22_nb501501_5,reducteur_de_debit_robinet_dia_24_nb501501_5,reducteur_de_debit_robinet_dia_22_nb501500_5,reducteur_de_debit_robinet_dia_24,reducteur_de_debit_neoperl_cascade_slc_pca_dc_nb02741195,reserve_robinet,autre_materiel,total_materiel_nombre,categorie_sia_et_unite_de_reference,valeur_nuitees_an_nb_sieges_nb_repas_an_m2_sre_etc,nb_robinets_total_avant_intervention,debit_moyen_robinets_avant_intervention_litres_min,nb_douches_total_avant_intervention,debit_moyen_douches_avant_intervention_litres_min,year_brt,rue_et_nb_original,AddresseNPA,SITG_ADRESSE,SITG_NPA,SITG_COMMUNE,SITG_NOM_NPA,SITG_EGID,SITG_SCORE,Addresse_sans_NPA,commune,NPA-COMMUNE
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""G0001""","""Grande""","""1269817""","""Résidence Jura""","""Av. J-D Maillard, 7""","""Avenue Jacob-Daniel MAILLARD 7""",null,null,"""1021280""","""non""","""1021280""","""2018""",""">2007""","""Gaz naturel""","""751""","""763""","""781""","""836""","""840""","""850""","""808""","""824""","""-""",null,null,null,null,null,"""2018""","""EMS""","""2018""","""3320""","""Gaz naturel""","""635290""","""kWh""","""635290""","""0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""G0002""","""Grande""","""1087032""","""EMS Franchises""","""Cité Vieusseux, 10""","""Cité Vieusseux 10""",null,null,"""1015829""","""non""","""1015888""","""1998""","""1991-1999""","""CAD tarifé""","""1117""","""1198""","""1162""","""1167""","""1023""","""880""","""974""","""936""","""-""",null,null,null,null,null,"""2018""","""EMS""","""2018""","""3298""","""CAD tarifé""","""778""","""MWh""","""778000""","""0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""G0003""","""Grande""","""91228""","""Hôtel Tor""","""Rue Ami Lévrier, 3""","""Rue Ami-LÉVRIER 3""",null,"""chaufferie dans un autre bâtim…","""1010817""","""non""","""1010817""","""2014""",""">2007""","""Gaz naturel""","""484""","""593""","""528""","""572""","""530""","""475""","""483""","""493""","""-""",null,null,null,null,null,"""2018""","""Hôtellerie""","""2018""","""2722""","""Gaz naturel""","""352834""","""kWh""","""352834""","""0""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""G0004""","""Grande""","""1101205""","""Clinique de la Plaine""","""Rue Charles Humbert, 5""","""Rue Charles-Humbert 5""",null,null,"""1013036""","""oui""","""1013036""","""2003""","""2000-2007""","""Gaz naturel""","""546""","""536""","""566""","""526""","""531""","""534""","""525""","""526""","""-""",null,null,null,null,null,"""2018""","""Hôpital""","""2018""","""2138.2""","""Gaz naturel""","""27

In [5]:
rue_cols: list[str] = [
    c for c in df_all.columns if ("rue" in c.lower() or "adresse" in c.lower())
]
# remove some columns that are not relevant
removed_cols: list[str] = [
    "Rue des Savoises, 15",
    "adresse facture 1",
    "adresse facture 2",
    "adresse facture 3",
    "adresse facture 4",
    "adresse facture 5",
    "adresse facture 6",
    "adresse facture 7",
    "analyse_adresse_inconnues",
    "analyse_adresse",
    "Adresse facturation",
    "Adresse LC",
]
rue_cols: list[str] = [c for c in rue_cols if c not in removed_cols]

print(rue_cols)

['Rue et numéro renseignés', 'rue SITG', 'ADRESSE', 'ADRESSE_BATIMENT', 'Rue et N°', 'Adresse SITG', 'adresse_spotfire', 'Adresse lieu de consommation', 'rue_et_nb', 'rue_et_nb_original', 'SITG_ADRESSE']


In [6]:
# Stack every rue/adresse column into a single column (vertical concat),
# then drop nulls. Each column is renamed to "adresse" first so they align
# under one name; order of columns/rows doesn't matter since we're just
# pooling all the non-null address values together.
df_adresse = (
    pl.concat(
        [df_all.select(pl.col(c).alias("adresse")) for c in rue_cols],
        how="vertical",
    )
    .drop_nulls()
    .unique()
)

print(f"Total non-null address values: {df_adresse.height}")
display(df_adresse)

Total non-null address values: 129968


adresse
str
"""Rte de Saconnex-d'Arve 129"""
"""Chemin Agénor- PARMELIN, 3BIS"""
"""Place du Temple, 1"""
"""Av. du Mervelet 14"""
"""Chemin des Rubiettes 16"""
…
"""Route Jean-Jacques- RIGAUD, 8"""
"""Chemin de l'Eglise 2A"""
"""Route d' Annecy, 140"""


In [10]:
df_adresse_existant: DataFrame = pl.read_csv(
    source="../tests/test_adresses.csv"
).select(pl.col(name="Rue et N°").alias(name="adresse"))

display(df_adresse_existant)

adresse
str
"""12 rue Jean-Charles Amat"""
"""37, sous Garan"""
"""Ancienne route, 78"""
"""Av de Thonex 30"""
"""Av. Blanc, 5"""
…
"""Ruelle du Midi 5"""
"""Vieux chemin d'Onex, 54"""
"""place Isaac Mercier, 1"""


In [13]:
df_final: DataFrame = (
    pl.concat(items=[df_adresse_existant, df_adresse]).unique().drop_nulls()
)

display(df_final)

csv_name: str = (
    f"../tests/test_adresses_{datetime.now().strftime(format='%Y%m%d_%H%M%S')}.csv"
)

df_final.write_csv(file=csv_name, separator=";")

adresse
str
"""Route de Frontenex 46"""
"""Chemin de Tattes-Fontaine, 43"""
"""Rue Ancienne 69"""
"""Rue des Peupliers, 24"""
"""Rte de La-Capite 173 A"""
…
"""Route du Mandement 539"""
"""Ch. des Bélosses 9"""
"""Ch. de Delay 7"""
